In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# OpenQASM 2 وحزمة Qiskit SDK

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
```
</details>

توفر حزمة Qiskit SDK بعض الأدوات للتحويل بين تمثيلات OpenQASM للبرامج الكمية وفئة [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit).

<span id="qasm2-import"></span>
## استيراد برنامج OpenQASM 2 إلى Qiskit
هناك دالتان لاستيراد برامج OpenQASM 2 إلى Qiskit.
وهما [`qasm2.load()`](../api/qiskit/qasm2#load) التي تأخذ اسم ملف، و[`qasm2.loads()`](../api/qiskit/qasm2#loads) التي تأخذ برنامج OpenQASM 2 كنص (string).

In [1]:
import qiskit.qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";
    qreg q[2];
    creg c[2];

    h q[0];
    cx q[0], q[1];

    measure q -> c;
"""
circuit = qiskit.qasm2.loads(program)
circuit.draw()

     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 

راجع [واجهة برمجة تطبيقات OpenQASM 2 في Qiskit](https://docs.quantum.ibm.com/api/qiskit/qasm2) لمزيد من المعلومات.

### استيراد برامج بسيطة
بالنسبة لمعظم برامج OpenQASM 2، يمكنك استخدام `qasm2.load` و`qasm2.loads` بسيط مع وسيط واحد فقط.

#### مثال: استيراد برنامج OpenQASM 2 كنص
استخدم `qasm2.loads()` لاستيراد برنامج OpenQASM 2 كنص إلى QuantumCircuit:

In [2]:
from qiskit import qasm2

program = """
    OPENQASM 2.0;
    include "qelib1.inc";

    qreg q[4];
    creg c[4];

    h q[0];
    cx q[0], q[1];

    // 'rxx' is not actually in `qelib1.inc`,
    // but Qiskit used to behave as if it were.
    rxx(0.75) q[2], q[3];

    measure q -> c;
"""
circuit = qasm2.loads(
    program,
    custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
)

#### Example: use a particular gate class when importing an OpenQASM 2 program

Qiskit cannot, in general, verify if the definition in an OpenQASM 2 `gate` statement corresponds exactly to a Qiskit standard-library gate.
Instead, Qiskit chooses a custom gate using the precise definition supplied.
This can be less efficient that using one of the built-in standard gates, or a user-defined custom gate.
You can manually define `gate` statements with particular classes.

In [3]:
from qiskit import qasm2
from qiskit.circuit import Gate
from qiskit.circuit.library import RZXGate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    # Link the OpenQASM 2 name 'my' with our custom gate.
    qasm2.CustomInstruction("my", 2, 1, MyGate),
    # Link the OpenQASM 2 name 'rzx' with Qiskit's
    # built-in RZXGate.
    qasm2.CustomInstruction("rzx", 1, 2, RZXGate),
]

program = """
    OPENQASM 2.0;

    gate my(theta, phi) q {
        U(theta / 2, phi, -theta / 2) q;
    }
    gate rzx(theta) a, b {
        // It doesn't matter what definition is
        // supplied, if the parameters match;
        // Qiskit will still use `RZXGate`.
    }

    qreg q[2];
    my(0.25, 0.125) q[0];
    rzx(pi) q[0], q[1];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

#### مثال: استيراد برنامج OpenQASM 2 من ملف
استخدم `load()` لاستيراد برنامج OpenQASM 2 من ملف إلى QuantumCircuit:

In [4]:
from qiskit import qasm2
from qiskit.circuit import Gate


# Define a custom gate that takes one qubit and two angles.
class MyGate(Gate):
    def __init__(self, theta, phi):
        super().__init__("my", 1, [theta, phi])


custom_instructions = [
    qasm2.CustomInstruction("my", 2, 1, MyGate, builtin=True),
]

program = """
    OPENQASM 2.0;
    qreg q[1];

    my(0.25, 0.125) q[0];
"""

circuit = qasm2.loads(
    program,
    custom_instructions=custom_instructions,
)

<span id="custom-instructions"></span>
### ربط بوابات OpenQASM 2 ببوابات Qiskit
افتراضيًا، يتعامل مستورد OpenQASM 2 في Qiskit مع ملف الإدراج `"qelib1.inc"` باعتباره مكتبة قياسية *فعلية*.
يتعامل المستورد مع هذا الملف على أنه يحتوي تحديدًا على البوابات الموصوفة في [الورقة البحثية الأصلية التي تُعرّف OpenQASM 2](https://arxiv.org/abs/1707.03429).
سيستخدم Qiskit البوابات المدمجة في [مكتبة الدوائر](../api/qiskit/circuit_library) لتمثيل البوابات الموجودة في `"qelib1.inc"`.
البوابات المعرَّفة في البرنامج عبر عبارات `gate` يدوية في OpenQASM 2 ستُبنى افتراضيًا كـ[فئات فرعية مخصصة من `Gate` في Qiskit](../api/qiskit/qiskit.circuit.Gate).

يمكنك إخبار المستورد باستخدام فئات [`Gate`](../api/qiskit/qiskit.circuit.Gate) محددة لعبارات `gate` التي يصادفها.
يمكنك أيضًا استخدام هذه الآلية لمعاملة أسماء بوابات إضافية على أنها "مدمجة"، أي لا تحتاج إلى تعريف صريح.
إذا حددت فئات البوابات المراد استخدامها لعبارات `gate` خارج `"qelib1.inc"`، فإن الدائرة الناتجة ستكون عادةً أكثر كفاءة للتعامل معها.

> **Warning:** اعتبارًا من Qiskit SDK v1.0، لا يزال *مُصدِّر* OpenQASM 2 في Qiskit (انظر [تصدير دائرة Qiskit إلى OpenQASM 2](#qasm2-export)) يتصرف كما لو أن `"qelib1.inc"` يحتوي على بوابات أكثر مما يحتوي عليه فعلًا.
> هذا يعني أن الإعدادات الافتراضية للمستورد قد لا تتمكن من استيراد برنامج صادر من مُصدِّرنا.
> راجع [المثال المحدد للتعامل مع المُصدِّر القديم](#qasm2-import-legacy) لحل هذه المشكلة.
> 
> هذا التناقض هو سلوك قديم في Qiskit، [وسيُحل في إصدار لاحق منه](https://github.com/Qiskit/qiskit/issues/10737).

لتمرير معلومات حول تعليمة مخصصة إلى مستورد OpenQASM 2، استخدم [فئة `qasm2.CustomInstruction`](../api/qiskit/qasm2#qiskit.qasm2.CustomInstruction).
تتطلب هذه الفئة أربع قطع معلومات مطلوبة، بالترتيب:

* **اسم** البوابة المستخدم في برنامج OpenQASM 2
* **عدد معاملات الزاوية** التي تأخذها البوابة
* **عدد الكيوبتات** التي تعمل عليها البوابة
* فئة أو دالة Python **المُنشِئة** للبوابة، التي تأخذ معاملات البوابة (دون الكيوبتات) كوسائط منفردة

إذا صادف المستورد تعريف `gate` يطابق تعليمة مخصصة معطاة، فسيستخدم تلك المعلومات المخصصة لإعادة بناء كائن البوابة.
إذا صودفت عبارة `gate` تطابق `name` تعليمة مخصصة، لكنها لا تطابق عدد المعاملات ولا عدد الكيوبتات، فسيُطلق المستورد خطأ [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror) للإشارة إلى عدم التطابق بين المعلومات المُقدَّمة والبرنامج.

بالإضافة إلى ذلك، يمكن تعيين وسيط خامس `builtin` اختياريًا إلى `True` لجعل البوابة متاحة تلقائيًا داخل برنامج OpenQASM 2، حتى لو لم تُعرَّف صراحةً.
إذا صادف المستورد تعريف `gate` صريحًا لتعليمة مخصصة مدمجة، فسيقبله بصمت.
كما سبق، إذا كان التعريف الصريح لنفس الاسم غير متوافق مع التعليمة المخصصة المُقدَّمة، فسيُطلق [`QASM2ParseError`](../api/qiskit/qasm2#qasm2parseerror).
هذا مفيد للتوافق مع مُصدِّرات OpenQASM 2 القديمة، ومع بعض منصات الحوسبة الكمية الأخرى التي تعامل "بوابات الأساس" في عتادها كتعليمات مدمجة.

يوفر Qiskit سمة بيانات للتعامل مع برامج OpenQASM 2 الصادرة من إصدارات قديمة من [قدرات تصدير OpenQASM 2 في Qiskit](#qasm2-export).
وهي [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility)، التي يمكن تمريرها كوسيط `custom_instructions` إلى [`qasm2.load()`](../api/qiskit/qasm2#load) و[`qasm2.loads()`](../api/qiskit/qasm2#loads).

<span id="qasm2-import-legacy"></span>
#### مثال: استيراد برنامج أنشأه مُصدِّر Qiskit القديم
يستخدم برنامج OpenQASM 2 هذا بوابات غير موجودة في النسخة الأصلية من `"qelib1.inc"` دون الإعلان عنها، لكنها بوابات قياسية في مكتبة Qiskit.
يمكنك استخدام [`qasm2.LEGACY_CUSTOM_INSTRUCTIONS`](../api/qiskit/qasm2#legacy-compatibility) لإخبار المستورد بسهولة باستخدام نفس مجموعة البوابات التي كان مُصدِّر OpenQASM 2 في Qiskit يستخدمها سابقًا.

In [5]:
import math
from qiskit import qasm2

program = """
    include "qelib1.inc";
    qreg q[2];
    rx(arctan(pi, 3 + add_one(0.2))) q[0];
    cx q[0], q[1];
"""


def add_one(x):
    return x + 1


customs = [
    # Our `add_one` takes only one parameter.
    qasm2.CustomClassical("add_one", 1, add_one),
    # `arctan` takes two parameters, and `math.atan2` implements it.
    qasm2.CustomClassical("arctan", 2, math.atan2),
]
circuit = qasm2.loads(program, custom_classical=customs)

#### مثال: استخدام فئة بوابة محددة عند استيراد برنامج OpenQASM 2
لا يستطيع Qiskit، بشكل عام، التحقق مما إذا كان التعريف في عبارة `gate` في OpenQASM 2 يتوافق تمامًا مع بوابة من مكتبة Qiskit القياسية.
بدلًا من ذلك، يختار Qiskit بوابة مخصصة باستخدام التعريف الدقيق المُقدَّم.
قد يكون هذا أقل كفاءة من استخدام إحدى البوابات القياسية المدمجة أو بوابة مخصصة يُعرِّفها المستخدم.
يمكنك تعريف عبارات `gate` يدويًا بفئات محددة.

In [6]:
from qiskit import QuantumCircuit, qasm2

# Define any circuit.
circuit = QuantumCircuit(2, 2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure([0, 1], [0, 1])

# Export to a string.
program = qasm2.dumps(circuit)

# Export to a file.
qasm2.dump(circuit, "my_file.qasm")

#### مثال: تعريف بوابة مدمجة جديدة في برنامج OpenQASM 2
إذا تم تعيين الوسيط `builtin=True`، فلا تحتاج البوابة المخصصة إلى تعريف مرتبط بها.